# Deliverable 4 - INF201

**Group members:** Akvile , Herman

**GitHub repository:** https://github.com/akvile-bot/Deliverable-4-.git

This notebook contains our solutions to Task 1 (`Temperature`) and Task 2 (`Fraction`).
The classes are in `temperature.py` and `fraction.py`, and the pytest tests are in
`test_temperature.py` and `test_fraction.py`. Below we explain our design choices,
show a short demo of each class and include the output from running pytest.

## Task 1: Temperatures

### Design choices
* **Validation:** The unit is converted to upper case, so `"c"` and `"C"` are both accepted. Any other unit raises `ValueError`. The value is converted to Kelvin, and if it is below 0 K (minus the tolerance) we raise `ValueError`. A non-numeric value raises `TypeError`. The default unit is Celsius.
* **Read-only attributes:** `value` and `unit` are stored in the private attributes `_value` and `_unit` and exposed through `@property` without setters. We also use `__slots__`, so new attributes cannot be added to an instance.
* **Tolerance:** The class constant `TOLERANCE = 1e-9` is used both in the absolute-zero check and in the comparisons. Without it, `Temperature(-459.67, "F")` would be rejected, because the conversion gives about `-5.7e-14 K` due to floating-point rounding.
* **Comparisons:** All six comparisons convert both temperatures to Kelvin, so different units can be compared directly. `==` is true when the difference is at most `1e-9`, and `<`/`>` require a difference larger than the tolerance. Comparing with a non-`Temperature` returns `NotImplemented`, so Python falls back to its default behaviour (`==` gives `False`, `<` raises `TypeError`).
* **Hashing:** Equality with a tolerance cannot guarantee that equal objects get equal hashes. We therefore set `__hash__ = None` and make `Temperature` unhashable on purpose.
* **Formatting:** `__str__` uses two decimals. Kelvin is written without the degree sign (`273.15 K`), since that is the SI convention.

In [1]:
%pip install pytest

Note: you may need to restart the kernel to use updated packages.


In [2]:
from temperature import Temperature

body = Temperature(37, "c")  # unit is case-insensitive
print(repr(body))
print(body)
print(body.convert_to("F"), "|", body.convert_to("K"))

# Cross-unit comparison
print("0 °C == 32 °F:", Temperature(0, "C") == Temperature(32, "F"))
print("Sorted:", sorted([Temperature(100, "C"), Temperature(0, "K"), Temperature(50, "F")]))

# Below absolute zero raises ValueError
try:
    Temperature(-300, "C")
except ValueError as err:
    print("ValueError:", err)

Temperature(37.0, 'C')
37.00 °C
98.60 °F | 310.15 K
0 °C == 32 °F: True
Sorted: [Temperature(0.0, 'K'), Temperature(50.0, 'F'), Temperature(100.0, 'C')]
ValueError: -300.0 C is below absolute zero (0 K)


In [ ]:
###Test output for Task 1
import pytest
pytest.main(["test_temperature.py", "-v"])

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.2.0, pluggy-1.7.0 -- /opt/anaconda3/envs/inf201/bin/python
cachedir: .pytest_cache
rootdir: /Users/akvileurbonaite/Deliverable-4-
plugins: platformdirs-4.13.0
collecting ... collected 79 items

test_temperature.py::test_default_unit_is_celsius PASSED                 [  1%]
test_temperature.py::test_units_are_case_insensitive[c-C] PASSED         [  2%]
test_temperature.py::test_units_are_case_insensitive[C-C] PASSED         [  3%]
test_temperature.py::test_units_are_case_insensitive[f-F] PASSED         [  5%]
test_temperature.py::test_units_are_case_insensitive[F-F] PASSED         [  6%]
test_temperature.py::test_units_are_case_insensitive[k-K] PASSED         [  7%]
test_temperature.py::test_units_are_case_insensitive[K-K] PASSED         [  8%]
test_temperature.py::test_value_is_stored_as_float PASSED                [ 10%]
test_temperature.py::test_attributes_are_

<ExitCode.OK: 0>

## Task 2: Fractions

### Design choices
* **Initialiser:** Both numerator and denominator must be `int` (`bool` is rejected even though it is a subclass of `int`), otherwise `TypeError` is raised. A zero denominator raises `ZeroDivisionError`.
* **Positive denominator:** Instead of raising an error for a negative denominator, we move the sign to the numerator, so `Fraction(1, -2)` is stored as `-1/2`. The denominator is therefore always strictly positive.
* **Lowest terms:** The fraction is reduced with `math.gcd`. Because every fraction has a unique reduced form, `==` can simply compare numerator and denominator.
* **Comparisons:** `<`, `<=`, `>`, `>=` use cross-multiplication, $\frac{a}{b} < \frac{c}{d} \iff ad < cb$, which is exact and works because both denominators are positive. Comparisons with `int` are supported; other types return `NotImplemented`.
* **Other special methods:** `__repr__`, `__str__`, `__add__`/`__radd__`, `__sub__`/`__rsub__`, `__mul__`/`__rmul__`, `__truediv__`/`__rtruediv__`, `__neg__`, `__abs__`, `__float__`, `__bool__` and `__hash__`. The reflected methods (`__radd__` etc.) make expressions like `1 + Fraction(1, 2)` work. `__hash__` gives `Fraction(2, 1)` the same hash as `2`, since they compare equal, so fractions can be used in sets and as dictionary keys.
* **Immutability:** The attributes are read-only properties, and every operation returns a new `Fraction`.

In [4]:
from fraction import Fraction

a, b = Fraction(1, 2), Fraction(2, -6)  # 2/-6 is stored as -1/3
print(repr(a), repr(b))
print(f"({a}) + ({b}) = {a + b}")
print(f"({a}) - ({b}) = {a - b}")
print(f"({a}) * ({b}) = {a * b}")
print(f"({a}) / ({b}) = {a / b}")
print("1 + a =", 1 + a, "| -b =", -b, "| abs(b) =", abs(b), "| float(a) =", float(a))
print("a > b:", a > b, "| Fraction(4, 2) == 2:", Fraction(4, 2) == 2)

# Equal fractions have the same hash, so the set keeps only one
print("Set of 1/2, 2/4, 3/6:", {Fraction(1, 2), Fraction(2, 4), Fraction(3, 6)})

Fraction(1, 2) Fraction(-1, 3)
(1/2) + (-1/3) = 1/6
(1/2) - (-1/3) = 5/6
(1/2) * (-1/3) = -1/6
(1/2) / (-1/3) = -3/2
1 + a = 3/2 | -b = 1/3 | abs(b) = 1/3 | float(a) = 0.5
a > b: True | Fraction(4, 2) == 2: True
Set of 1/2, 2/4, 3/6: {Fraction(1, 2)}


In [ ]:
### Test output for Task 2
import pytest
pytest.main(["test_fraction.py", "-v"])

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.2.0, pluggy-1.7.0 -- /opt/anaconda3/envs/inf201/bin/python
cachedir: .pytest_cache
rootdir: /Users/akvileurbonaite/Deliverable-4-
plugins: platformdirs-4.13.0
collecting ... collected 71 items

test_fraction.py::test_init_stores_numerator_and_denominator PASSED      [  1%]
test_fraction.py::test_default_denominator_is_one PASSED                 [  2%]
test_fraction.py::test_init_reduces_to_lowest_terms[2-4-expected0] PASSED [  4%]
test_fraction.py::test_init_reduces_to_lowest_terms[10-5-expected1] PASSED [  5%]
test_fraction.py::test_init_reduces_to_lowest_terms[0-7-expected2] PASSED [  7%]
test_fraction.py::test_init_reduces_to_lowest_terms[-6-9-expected3] PASSED [  8%]
test_fraction.py::test_init_reduces_to_lowest_terms[100-75-expected4] PASSED [  9%]
test_fraction.py::test_denominator_is_always_positive[1--2-expected0] PASSED [ 11%]
test_fraction.py::test_deno

<ExitCode.OK: 0>

## Use of generative AI

we used Claude AI as a supporting tool during the preparation and completion of this coding assignment. Before starting the assignment, we used AI to review the task description, break down the requirements, and explain what was expected of us.

Throughout the process, we also used AI to help us understand different coding concepts, explain unfamiliar code and functions, and clarify how specific parts of the code worked. Additionally, AI was used to help us identify potential errors, understand error messages, and explore possible ways to solve coding problems.

The purpose of using AI was to support our learning and improve our understanding of the programming concepts involved. All suggestions and explanations were reviewed and considered by us, and we made the decisions regarding the final implementation of the code.

In [6]:
print(open("git_log.txt").read()) 
#Shows our git log

* 143a7aa - Akvile Urbonaite, 2 minutes ago : Fixing the codes with comments and testing
* 772daf4 - Akvile Urbonaite, 22 minutes ago : Add requirements.txt
* e3155f9 - Akvile Urbonaite, 22 minutes ago : Add gitignore and remove pycache
* 29e6fd8 - Herman, 33 minutes ago : hei?
* ce1c8ac - Herman, 33 minutes ago : message
* 14abb48 - Herman, 37 minutes ago : yessir
* 384eca4 - AkvileUrbonaite, 13 hours ago : Add files via upload
* 4a1ec60 - AkvileUrbonaite, 13 hours ago : Add files via upload
* 39e8229 - AkvileUrbonaite, 13 hours ago : Initial commit
